In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [30]:
import numpy as np
import pandas as pd
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [3]:
own_dir = "/content/drive/MyDrive/Colab Notebooks/AIS-IRS/practice_module/stage1_student4/polyvore"
os.chdir(own_dir)

## padding & concatenation

In [6]:
cache = np.load("fashionclip_embeddings.npz")
embedding_dict = {product_id: cache[product_id] for product_id in cache.files}

df = pd.read_csv("clean_outfits_compatibility_prediction.csv")
df["products"] = df["products"].apply(json.loads)

max_items = df["products"].apply(len).max()
embedding_dim = 512

def get_outfit_embedding(products):
    embeddings = [embedding_dict[pid] for pid in products if pid in embedding_dict]

    # Padding with zero vectors
    while len(embeddings) < max_items:
        embeddings.append(np.zeros(embedding_dim))

    return np.concatenate(embeddings)

df["embedding"] = df["products"].apply(get_outfit_embedding)
df

,outfit_id,label,products,embedding
0,O0001,1,"[119704139_1, 119704139_2, 119704139_3, 119704...","[-0.36080819368362427, -0.05561429262161255, 0..."
1,O0002,1,"[119314458_1, 119314458_2, 119314458_5, 119314...","[-1.0425564050674438, 1.5769972801208496, -0.7..."
2,O0003,1,"[147361785_1, 147361785_2, 147361785_3, 147361...","[-0.021296493709087372, 0.0059826187789440155,..."
3,O0005,1,"[192637078_1, 192637078_2, 192637078_5, 192637...","[-0.10370905697345734, 0.2831333875656128, 0.3..."
4,O0006,1,"[163560873_1, 163560873_2, 163560873_3, 163560...","[-0.1646185964345932, -0.8418341279029846, -0...."
...,...,...,...,...
4130,O7072,0,"[184041913_1, 209222683_1, 207694450_3, 216428...","[-0.5749406814575195, 0.12060105800628662, -0...."
4131,O7073,0,"[209709393_3, 212653508_4, 181715094_2, 137945...","[-0.1585898995399475, 0.4719929099082947, -0.7..."
4132,O7074,0,"[109684687_6, 215842725_3, 155392434_5]","[-0.28845033049583435, 0.7157207727432251, -1...."
4133,O7075,0,"[207577310_1, 216381976_2, 130865354_3, 216900...","[-0.7668565511703491, 0.03206813335418701, -0...."


## model train

In [9]:
X = np.stack(df["embedding"].values)
y = df["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Split temporary data into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (2894, 4096)
Validation: (620, 4096)
Test: (621, 4096)


In [28]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    alpha=0.001,
    max_iter=300,
    random_state=42
)

mlp.fit(X_train, y_train)

MLPClassifier(alpha=0.001, hidden_layer_sizes=(128, 64), max_iter=300,
              random_state=42)

In [29]:
val_pred = mlp.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print("Validation Classification Report:")
print(classification_report(y_val, val_pred, target_names=["Incompatible", "Compatible"]))

test_pred = mlp.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, test_pred))
print("Test Classification Report:")
print(classification_report(y_test, test_pred, target_names=["Incompatible", "Compatible"]))

Validation Accuracy: 0.7838709677419354
Validation Classification Report:
              precision    recall  f1-score   support

Incompatible       0.82      0.79      0.81       351
  Compatible       0.74      0.77      0.76       269

    accuracy                           0.78       620
   macro avg       0.78      0.78      0.78       620
weighted avg       0.79      0.78      0.78       620


Test Accuracy: 0.7246376811594203
Test Classification Report:
              precision    recall  f1-score   support

Incompatible       0.77      0.73      0.75       352
  Compatible       0.67      0.72      0.69       269

    accuracy                           0.72       621
   macro avg       0.72      0.72      0.72       621
weighted avg       0.73      0.72      0.73       621



In [31]:
joblib.dump(mlp, "outfits_compatibility_mlp.pkl")

['outfits_compatibility_mlp.pkl']